In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from visualisation import *
from plotly.subplots import make_subplots

from methods.geodesics import geodesic_distances_function
from generate_data import gen_2d_data, load_image_point_cloud
from diffusion_geometry import DiffusionGeometry

Create some data

In [ ]:
data1 = load_image_point_cloud("../data/data12.jpeg", n=800, threshold=0.7, intensity_weighted=False)
data1 = -data1[:, [1,0]]  # Swap columns for correct orientation
plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1)

In [ ]:
index = 6
# dist, v = geodesic_distances_function(dg1, index, reg=True, num_subsample=50)
dist, v = geodesic_distances_function(dg1, index)
fig = plot_scatter_2d(data1, color = dist - dist.max()/2)
fig.add_trace(go.Scatter(
    x=[data1[index, 0]],
    y=[data1[index, 1]],
    mode='markers',
    marker=dict(color='black', size=15),
))
fig.show()

In [ ]:
indices_2d = [0,4,6]

In [ ]:
from generate_data import gen_3d_data
data2, _ = gen_3d_data(kind = "swiss_roll", n = 3000, noise = 0.0)
# data2, _ = gen_3d_data(kind = "swiss_roll", n = 1500, noise = 0.0)
dg2 = DiffusionGeometry.from_point_cloud(data2)

camera = dict(eye=dict(x=-0.2, y=-1.7, z=0.5), center=dict(x=0.04, y=0, z=-0.08))
plot_scatter_3d(data2, color = 'black', camera=camera).show()

In [ ]:
index = 1
# dist, v = geodesic_distances_function(dg2, index, reg=True, num_subsample=50)
dist, v = geodesic_distances_function(dg2, index)
fig = plot_scatter_3d(data2, color = dist - dist.max()/2, camera=camera)
fig.add_trace(go.Scatter3d(
    x = [data2[index, 0]],
    y = [data2[index, 1]],
    z = [data2[index, 2]],
    mode = 'markers',
    marker = dict(
        color = 'black',
        # color = '#e32636',
        size = 10,
        # symbol = 'x',
        opacity = 1.0
    ),
))
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Define which subplots are 3D
specs = [
    [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
    [{"type": "scene"}, {"type": "scene"}, {"type": "scene"}],
]

fig = make_subplots(
    rows=2,
    cols=3,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.01,
    row_heights=[0.45, 0.55],
)

dists_2d = []
for i, idx in enumerate(indices_2d):
    # dist, _ = geodesic_distances_function(dg1, idx, reg=True, num_subsample=50)
    dist, _ = geodesic_distances_function(dg1, idx)
    dists_2d.append(dist)
amax_2d = max([max(d) for d in dists_2d])

figs_2d = []
for i, idx in enumerate(indices_2d):
    f = plot_scatter_2d(data1, 
                        color = -dists_2d[i] + amax_2d/2, 
                        amax=amax_2d/2,
                        size=4)
    f.add_trace(go.Scatter(
        x=[data1[idx, 0]],
        y=[data1[idx, 1]],
        mode='markers',
        marker=dict(color='black', size=12),
    ))
    figs_2d.append(f)
    fig.add_traces(list(f.data), rows=[1]*len(f.data), cols=[i+1]*len(f.data))

dists_3d = []
for i, idx in enumerate(indices_3d):
    dist, _ = geodesic_distances_function(dg2, idx, reg=True, num_subsample=500)
    # dist, _ = geodesic_distances_function(dg2, idx)
    dists_3d.append(dist)
amax_3d = max([max(d) for d in dists_3d])

figs_3d = []
for i, idx in enumerate(indices_3d):
    f = plot_scatter_3d(data2, 
                        color = -dists_3d[i] + amax_3d/2, 
                        camera=camera, 
                        amax=amax_3d/2,
                        size=2)
    f.add_trace(go.Scatter3d(
        x=[data2[idx, 0]],
        y=[data2[idx, 1]],
        z=[data2[idx, 2]],
        mode='markers',
        marker=dict(color='black', size=7),
    ))
    figs_3d.append(f)
    fig.add_traces(list(f.data), rows=[2]*len(f.data), cols=[i+1]*len(f.data))

# Synchronize axis ranges for top row (2D plots)
x_vals = []
y_vals = []
for f in figs_2d:
    for t in f.data:
        if hasattr(t, 'x') and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, 'y') and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])

xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]

# Apply to all top row subplots
fig.update_xaxes(range=xrange, row=1)
fig.update_yaxes(range=yrange, row=1)

# Synchronize axis ranges for bottom row (3D plots)
x_vals_3d = []
y_vals_3d = []
z_vals_3d = []
for f in figs_3d:
    for t in f.data:
        if hasattr(t, 'x') and t.x is not None:
            x_vals_3d.extend([v for v in t.x if v is not None])
        if hasattr(t, 'y') and t.y is not None:
            y_vals_3d.extend([v for v in t.y if v is not None])
        if hasattr(t, 'z') and t.z is not None:
            z_vals_3d.extend([v for v in t.z if v is not None])

xrange_3d = [min(x_vals_3d), max(x_vals_3d)]
yrange_3d = [min(y_vals_3d), max(y_vals_3d)]
zrange_3d = [min(z_vals_3d), max(z_vals_3d)]

# Apply to all bottom row 3D subplots
for i in range(1, 5):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        key: dict(
            camera=camera,
            xaxis=dict(range=xrange_3d),
            yaxis=dict(range=yrange_3d),
            zaxis=dict(range=zrange_3d)
        )
    })

fig.update_layout(width=1000, height=600)
clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:

rows, cols = 2, 4
fig_w, fig_h = 1000, 800

def labels(row, col):
    if col == 1:
        return rf"$X$"
    elif col == 2:
        return "$\|X\| = \sqrt{g(X,X)}$"
    elif col == 3:
        return rf"$Y$"
    elif col == 4:
        return rf"$g(X,Y)$"
    else:
        return ""

overpic_labels(fig, labels, 
               stretch_x=0.98, 
               stretch_y=0.96,
               offset_y=-1.0)


In [ ]:
index = 0
dist, v = geodesic_distances_function(dg2, index, reg=True, num_subsample=80)
# dist, v = geodesic_distances_function(dg2, index)
fig = plot_scatter_3d(data2, color = dist - dist.max()/2, camera=camera)
fig.add_trace(go.Scatter3d(
    x = [data2[index, 0]],
    y = [data2[index, 1]],
    z = [data2[index, 2]],
    mode = 'markers',
    marker = dict(
        color = 'black',
        # color = '#e32636',
        size = 10,
        # symbol = 'x',
        opacity = 1.0
    ),
))
fig.show()


In [ ]:


fig = plot_scatter_3d(data2, color = (dist < dist[np.argsort(dist)[300]]) * (dist - dist.max()/2), camera=camera)
fig.add_trace(go.Scatter3d(
    x = [data2[index, 0]],
    y = [data2[index, 1]],
    z = [data2[index, 2]],
    mode = 'markers',
    marker = dict(
        color = 'black',
        # color = '#e32636',
        size = 10,
        # symbol = 'x',
        opacity = 1.0
    ),
))
fig.show()

In [ ]:
from tqdm import tqdm
dg = dg2
data = dg.backend.data_matrix


ambient_dist_pointwise = np.linalg.norm(data - data[index], axis=1)
ambient_dist_pointwise = np.maximum(ambient_dist_pointwise, 1e-6)

# derivative of ambient distance function in the ambient space
# d(d_y(x)) = x-y / ||x-y||
d_ambient = (data[index] - data) / ambient_dist_pointwise[:, np.newaxis]
# project to tangent space at x (multiply by Gamma_x)
d_ambient = np.einsum('ab,pb->pa', dg.backend.gamma_ambient[index], d_ambient)

logs = []
# randomly subsample log indices
log_indices = np.argsort(dist)[:50]
for i in tqdm(log_indices):
    # distance function from y. d_y
    dist_i, v_i = geodesic_distances_function(dg, i, reg=True, num_subsample=120)
    # derivative of d_y at x
    d_dist_i = v_i.grad().to_ambient()[index] + d_ambient[i]
    logs.append(-0.5 * dist[i] * d_dist_i)
log = np.array(logs)
log *= -0.5


# d_dist = v.grad().to_ambient() + d_ambient
# plot_quiver_3d(data, d_dist, scale=1, arrow_scale=1, line_width=2).show()

# log = d_dist * dist[:, np.newaxis]
# plot_quiver_3d(data, log, scale=0.1, arrow_scale=1, line_width=2).show()

In [ ]:
# from sklearn.decomposition import PCA
# pca = PCA(n_components=2)
# geodesic_normal_coords = pca.fit_transform(log)
geodesic_normal_coords = np.linalg.svd(log, full_matrices=False)[0][:, :2]
# plot_scatter_2d(geodesic_normal_coords[], color='black', size=5).show()

fig = plot_scatter_3d(data2, color = 'lightgray', size = 2, camera=camera)
fig.add_trace(go.Scatter3d(
    x = data2[log_indices, 0],
    y = data2[log_indices, 1],
    z = data2[log_indices, 2],
    mode = 'markers',
    marker = dict(
        color = geodesic_normal_coords[:, 1],
        size = 5,
        opacity = 1.0,
        cmid=0
    ),
))
fig.add_trace(go.Scatter3d(
    x = [data2[index, 0]],
    y = [data2[index, 1]],
    z = [data2[index, 2]],
    mode = 'markers',
    marker = dict(
        color = 'black',
        size = 10,
        opacity = 1.0
    ),
))
fig.show()
